In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.store.base import BaseStore
from typing import TypedDict, Annotated
import os 
os.environ["HF_HOME"] = "E:\\huggingface_embedding_cache"
from langchain_huggingface import HuggingFaceEmbeddings



In [ ]:
model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True},
)

store = InMemoryStore(index={
    "dims": 1536,
    "embed": model
})
saver = InMemorySaver()
class State(TypedDict):
    question: str

namespace = ('user', 'nadeem')

def add_memory(state: State, store: BaseStore):
   store.put(namespace, 
          'favourite Sport',
          {"data": "Nadeem likes to play cricket"}
        )
store.put(namespace, 
          'hobby',
          {"data": "Nadeems hobby is gym"}
        )
store.put(namespace, 
          'food',
          {"data": "Nadeem likes to eat eggs"}
        )
store.put(namespace, 
          'today',
          {"data": "Nadeem will watch football final today"}
        )


def fetch_memories(state: State, store: BaseStore):
    all_memories = store.search(('user', 'nadeem'), query='what is user doing today', limit=3)
    for memory in all_memories:
        print(memory)
    return {}


graph = StateGraph(State)


graph.add_node('chat_node', add_memory)
graph.add_node('expl_node', fetch_memories)

graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', 'expl_node')
graph.add_edge('expl_node', END)

workflow = graph.compile(checkpointer=saver, store=store)

result = workflow.invoke({'question':""})
# print(result)
